# Olist 电商履约与客户体验诊断

## tl;dr

已交付 96,478 单；延迟率 6.77%。履约与评分均有效的订单中，延迟组低评分率 62.42%，按时组 9.27%。这是观察性关联，不能推断因果。

## Context & Methods

来源：[Kaggle Olist v2](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)。按购买日期归属，历史最终状态。

### Key Assumptions

- 商品金额仅汇总 price，不含运费；BRL，不代表平台收入或利润。
- 评分按最新回答、创建时间、评价 ID 取一条；缺失时间排后。
- 交付比较日历日期，来源本地时区不明，不作 UTC 转换。
- 原始数据需先按 README 下载并映射；在项目根目录启动／执行 notebook。
- 文件由标准 Python 顺序执行代码单元并保存输出；本次没有启动 Jupyter 内核。代码不含 magic 或内核专属操作，可安装 Jupyter 后用 nbconvert 再执行。

## Data

### 1. 读取规范化 CSV 与已验证快照

原始文件哈希保存在 data/raw/kaggle-v2/provenance.json。只读取所需三表。

In [1]:
from pathlib import Path
from collections import defaultdict
from decimal import Decimal
import csv, json
root = Path.cwd()
if not (root / 'data/canonical/olist-v2').exists():
    root = root.parent
def read_table(name):
    with (root / 'data/canonical/olist-v2' / (name + '.csv')).open(encoding='utf-8', newline='') as stream:
        return list(csv.DictReader(stream))
orders = read_table('orders')
items = read_table('order_items')
reviews = read_table('order_reviews')
saved = json.loads((root / 'output/analysis/case.json').read_text(encoding='utf-8'))
print({'orders': len(orders), 'items': len(items), 'reviews': len(reviews), 'version': saved['id']})

{'orders': 99441, 'items': 112650, 'reviews': 99224, 'version': 'ab7a405f82b7cf35559a7bea'}


### 2. 确定粒度与评价选择

商品按订单求和；评价独立按订单排序，避免两个明细表相乘。

In [2]:
assert len({row['order_id'] for row in orders}) == len(orders)
order_ids = {row['order_id'] for row in orders}
assert all(row['order_id'] in order_ids for row in items + reviews)
amounts = defaultdict(Decimal)
for row in items:
    amounts[row['order_id']] += Decimal(row['price'])
review_rows = sorted(reviews, key=lambda row: (row['review_answered_at'], row['review_created_at'], row['review_id']))
latest = {row['order_id']: row for row in review_rows}
print({'item_orders': len(amounts), 'reviewed_orders_all_status': len(latest)})

{'item_orders': 98666, 'reviewed_orders_all_status': 98673}


## Results

### 3. 独立计算核心分子分母

零分母应显示不适用；缺评分、缺交付日期分别保留与披露。

In [3]:
metrics = dict(total_orders=len(orders), delivered_orders=0, amount_cents=0, delivery_eligible=0, late_orders=0, review_eligible=0, low_reviews=0)
association = {0: {'n': 0, 'low': 0}, 1: {'n': 0, 'low': 0}}
strata = defaultdict(lambda: {0: {'n': 0, 'low': 0}, 1: {'n': 0, 'low': 0}})
for row in orders:
    if row['order_status'] != 'delivered':
        continue
    metrics['delivered_orders'] += 1
    metrics['amount_cents'] += int(amounts[row['order_id']] * 100)
    valid_dates = bool(row['delivered_at'] and row['estimated_delivery_at'])
    score = latest.get(row['order_id'], {}).get('review_score')
    if valid_dates:
        late = int(row['delivered_at'][:10] > row['estimated_delivery_at'][:10])
        metrics['delivery_eligible'] += 1
        metrics['late_orders'] += late
    if score:
        low = int(int(score) <= 2)
        metrics['review_eligible'] += 1
        metrics['low_reviews'] += low
    if valid_dates and score:
        association[late]['n'] += 1
        association[late]['low'] += low
        group = strata[(row['purchased_at'][:7], row['customer_state'])][late]
        group['n'] += 1
        group['low'] += low
assert all(saved['metrics'][key] == value for key, value in metrics.items())
print(metrics)
print({'late_rate': metrics['late_orders'] / metrics['delivery_eligible'], 'low_rate': metrics['low_reviews'] / metrics['review_eligible']})

{'total_orders': 99441, 'delivered_orders': 96478, 'amount_cents': 1322149811, 'delivery_eligible': 96470, 'late_orders': 6534, 'review_eligible': 95832, 'low_reviews': 12273}
{'late_rate': 0.06773090079817559, 'low_rate': 0.12806786877034812}


### 4. 复查分层结果

每个购买月份 × 地区分层中，两组均至少 30 单；按该层总订单数加权。此阈值是探索性展示规则，不是因果识别依据。

In [4]:
eligible = [pair for pair in strata.values() if pair[0]['n'] >= 30 and pair[1]['n'] >= 30]
covered = sum(pair[0]['n'] + pair[1]['n'] for pair in eligible)
weighted = sum((pair[1]['low'] / pair[1]['n'] - pair[0]['low'] / pair[0]['n']) * (pair[0]['n'] + pair[1]['n']) for pair in eligible) / covered
assert covered == saved['stratified']['orders']
assert abs(weighted - saved['stratified']['weighted_difference']) < 1e-12
print({'association': association, 'common_strata': len(eligible), 'covered_orders': covered, 'weighted_difference': weighted})

{'association': {0: {'n': 89443, 'low': 8289}, 1: {'n': 6381, 'low': 3983}}, 'common_strata': 49, 'covered_orders': 49990, 'weighted_difference': 0.482606021265764}


## Takeaways

延迟组的低评分关联值得排查。建议以高影响地区为起点，核对卖家、品类和承诺时间，设计有对照的履约试点。

分层结果仅覆盖交叉样本的约 52.17%，仍可能受商品、金额、卖家及评价意愿影响。历史末期不完整，缺少流量与完整成本，不计算转化率、利润或已实现提升。详见正式 PDF 和工作台质量页面。